In [14]:
import sympy as sp
from sympy.combinatorics import permutations
import itertools
from dataclasses import dataclass
from typing import Literal

In [42]:
@dataclass
class FermionOperator:
    index: int
    spin: str
    dagger: bool = True
    def __init__(self, index, spin, dagger=True):
        if spin not in ['up', 'down', '↑', '↓']:
            raise ValueError('spin must be "up" or "down"')
        self.index = index
        self.spin = {'up': '↑', 'down': '↓'}.get(spin, spin)
        self.dagger = dagger
    def __str__(self):
        if self.dagger:
            dagger = '^\\dagger'
        else:
            dagger = ''
        return f'c{dagger}_{{{self.index}{self.spin}}}'

In [115]:
configuration = '↓↑↓↑↓↑↑↓↓↑'
term = [FermionOperator(i, spin) for i, spin in enumerate(configuration)]
assert configuration.count('↑') == configuration.count('↓')

In [116]:
def f(i: int, j: int):
    return sp.Symbol(f'f_{{{i}{j}}}')

In [117]:
expression = sp.sympify(0)
for permutation in itertools.permutations(range(len(term))):
    new_term = [term[i] for i in permutation]
    if all(c.spin == ['↑', '↓'][i % 2] for i, c in enumerate(new_term)):
        sign = permutations.Permutation(permutation).signature()
        expr_term = sign
        for ci, cj in zip(new_term[::2], new_term[1::2]):
            expr_term *= f(ci.index, cj.index)
        expression += expr_term

In [118]:
expression

120*f_{10}*f_{32}*f_{54}*f_{67}*f_{98} - 120*f_{10}*f_{32}*f_{54}*f_{68}*f_{97} - 120*f_{10}*f_{32}*f_{57}*f_{64}*f_{98} + 120*f_{10}*f_{32}*f_{57}*f_{68}*f_{94} + 120*f_{10}*f_{32}*f_{58}*f_{64}*f_{97} - 120*f_{10}*f_{32}*f_{58}*f_{67}*f_{94} - 120*f_{10}*f_{34}*f_{52}*f_{67}*f_{98} + 120*f_{10}*f_{34}*f_{52}*f_{68}*f_{97} + 120*f_{10}*f_{34}*f_{57}*f_{62}*f_{98} - 120*f_{10}*f_{34}*f_{57}*f_{68}*f_{92} - 120*f_{10}*f_{34}*f_{58}*f_{62}*f_{97} + 120*f_{10}*f_{34}*f_{58}*f_{67}*f_{92} + 120*f_{10}*f_{37}*f_{52}*f_{64}*f_{98} - 120*f_{10}*f_{37}*f_{52}*f_{68}*f_{94} - 120*f_{10}*f_{37}*f_{54}*f_{62}*f_{98} + 120*f_{10}*f_{37}*f_{54}*f_{68}*f_{92} + 120*f_{10}*f_{37}*f_{58}*f_{62}*f_{94} - 120*f_{10}*f_{37}*f_{58}*f_{64}*f_{92} - 120*f_{10}*f_{38}*f_{52}*f_{64}*f_{97} + 120*f_{10}*f_{38}*f_{52}*f_{67}*f_{94} + 120*f_{10}*f_{38}*f_{54}*f_{62}*f_{97} - 120*f_{10}*f_{38}*f_{54}*f_{67}*f_{92} - 120*f_{10}*f_{38}*f_{57}*f_{62}*f_{94} + 120*f_{10}*f_{38}*f_{57}*f_{64}*f_{92} - 120*f_{12}*f_{30

In [120]:
idx_up = [i for i, c in enumerate(term) if c.spin == '↑']
idx_down = [i for i, c in enumerate(term) if c.spin == '↓']

In [121]:
m = sp.Matrix([[f(j, i) for j in idx_up] for i in idx_down])

In [122]:
sp.simplify(expression / m.det())

120